## Ensemble model

### Loading data and importing libraries

In [1]:
# Ensemble models - combining simple classifiers

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# Load data
X_train = pd.read_csv('../data/X_train_9xQjqvZ.csv', index_col = 'ROW_ID')
X_test = pd.read_csv('../data/X_test_1zTtEnD.csv', index_col = 'ROW_ID')
y_train = pd.read_csv('../data/y_train_Ppwhaz8.csv', index_col = 'ROW_ID')

# Prepare data (keep SV1 and fill missing with 0)
X_base = X_train.fillna(0)
X_test_base = X_test.fillna(0)

# Convert target variable to binary
y = (y_train['target'] > 0).astype(int)

# Feature columns excluding labels - TS & ALLOCATION
feat_cols = [col for col in X_base.columns if col not in ['TS', 'ALLOCATION']]

print(f"Data loaded: {X_base.shape[0]} examples")
print(f"Features: {len(feat_cols)}")
print("Using only base features (no engineering to avoid overfitting)")

Data loaded: 527073 examples
Features: 42
Using only base features (no engineering to avoid overfitting)


### Defining models

In [3]:
# Defining 4 model constrained to prevent overfitting

models = {
    'Logistic Regression' : LogisticRegression(max_iter = 1000, random_state = 42),
    'Random Forest' : RandomForestClassifier(n_estimators = 50, max_depth = 5, random_state = 42),
    'LightGBM Simple' : lgb.LGBMClassifier(n_estimators = 50, learning_rate = 0.05, max_depth = 3, random_state = 42, verbose = -1),
    'XGBoost Simple' : xgb.XGBClassifier(n_estimators = 50, learning_rate = 0.05, max_depth = 3, random_state = 42, eval_metric = 'logloss')
}

print("Ensemble members:")
for name in models.keys():
    print(f" - {name}")

Ensemble members:
 - Logistic Regression
 - Random Forest
 - LightGBM Simple
 - XGBoost Simple


### Testing each model individually

In [7]:
print("\n" + "="*60)
print("Testing individual models with 5-fold CV")
print("="*60)

individual_scores = {}

for model_name, model in models.items():
    print(f"\nTesting {model_name}: ")

    kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_base), 1):
        X_train_fold = X_base.iloc[train_idx][feat_cols]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X_base.iloc[val_idx][feat_cols]
        y_val_fold = y.iloc[val_idx]

        # Train and predict
        model.fit(X_train_fold, y_train_fold)
        pred = model.predict(X_val_fold)
        acc = accuracy_score(y_val_fold, pred)
        scores.append(acc)
        print(f"Fold {fold}: {acc:.4f}")

    mean_score= np.mean(scores)
    individual_scores[model_name] = mean_score
    print(f"Mean: {mean_score:.4f} ± {np.std(scores):.4f}")

print("\n" + "="*60)
print("Individual model summary: ")
for name, score in sorted(individual_scores.items(), key = lambda x: x[1], reverse = True):
    print(f" {name:25s}: {score:.4f}")
print("="*60)


Testing individual models with 5-fold CV

Testing Logistic Regression: 
Fold 1: 0.5095
Fold 2: 0.5066
Fold 3: 0.5100
Fold 4: 0.5046
Fold 5: 0.5078
Mean: 0.5077 ± 0.0020

Testing Random Forest: 
Fold 1: 0.5273
Fold 2: 0.5239
Fold 3: 0.5260
Fold 4: 0.5258
Fold 5: 0.5269
Mean: 0.5260 ± 0.0012

Testing LightGBM Simple: 
Fold 1: 0.5263
Fold 2: 0.5212
Fold 3: 0.5230
Fold 4: 0.5260
Fold 5: 0.5236
Mean: 0.5240 ± 0.0019

Testing XGBoost Simple: 
Fold 1: 0.5261
Fold 2: 0.5216
Fold 3: 0.5234
Fold 4: 0.5262
Fold 5: 0.5240
Mean: 0.5243 ± 0.0017

Individual model summary: 
 Random Forest            : 0.5260
 XGBoost Simple           : 0.5243
 LightGBM Simple          : 0.5240
 Logistic Regression      : 0.5077


### Combining models (ensemble)

In [8]:
# Test ensemble: average predictions from all 4 models

print("\n" + "="*60)
print("Testing ensemble (average of all 4 models):")
print("="*60)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_base), 1):
    X_train_fold = X_base.iloc[train_idx][feat_cols]
    y_train_fold = y.iloc[train_idx]
    X_val_fold = X_base.iloc[val_idx][feat_cols]
    y_val_fold = y.iloc[val_idx]
    
    # Train all models
    predictions = []
    for model_name, model in models.items():
        model.fit(X_train_fold, y_train_fold)
        # Get probability predictions (not just 0/1)
        pred_proba = model.predict_proba(X_val_fold)[:, 1]
        predictions.append(pred_proba)
    
    # Average the probabilities
    avg_proba = np.mean(predictions, axis=0)
    # Convert to binary: >0.5 = predict 1, else 0
    ensemble_pred = (avg_proba > 0.5).astype(int)
    
    acc = accuracy_score(y_val_fold, ensemble_pred)
    ensemble_scores.append(acc)
    print(f"  Fold {fold}: {acc:.4f}")

ensemble_mean = np.mean(ensemble_scores)
print(f"  Mean: {ensemble_mean:.4f} ± {np.std(ensemble_scores):.4f}")

print("\n" + "="*60)
print("COMPARISON:")
print(f"  Best individual (RF):  {individual_scores['Random Forest']:.4f}")
print(f"  Ensemble (all 4):      {ensemble_mean:.4f}")
print(f"  Improvement:           {(ensemble_mean - individual_scores['Random Forest'])*100:+.2f}%")
print("="*60)


Testing ensemble (average of all 4 models):
  Fold 1: 0.5269
  Fold 2: 0.5223
  Fold 3: 0.5249
  Fold 4: 0.5240
  Fold 5: 0.5244
  Mean: 0.5245 ± 0.0015

COMPARISON:
  Best individual (RF):  0.5260
  Ensemble (all 4):      0.5245
  Improvement:           -0.15%


### Ensemble of just the tree models

In [ ]:
# Ensemble of just tree models (exclude Logistic Regression)

print("\n" + "="*60)
print("Testing ensemble (tree models only - no LogReg)")
print("="*60)

tree_models = {k: v for k, v in models.items() if k != 'Logistic Regression'}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
tree_ensemble_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_base), 1):
    X_train_fold = X_base.iloc[train_idx][feat_cols]
    y_train_fold = y.iloc[train_idx]
    X_val_fold = X_base.iloc[val_idx][feat_cols]
    y_val_fold = y.iloc[val_idx]
    
    # Train only tree models
    predictions = []
    for model_name, model in tree_models.items():
        model.fit(X_train_fold, y_train_fold)
        pred_proba = model.predict_proba(X_val_fold)[:, 1]
        predictions.append(pred_proba)
    
    # Average
    avg_proba = np.mean(predictions, axis=0)
    ensemble_pred = (avg_proba > 0.5).astype(int)
    
    acc = accuracy_score(y_val_fold, ensemble_pred)
    tree_ensemble_scores.append(acc)
    print(f"  Fold {fold}: {acc:.4f}")

tree_ensemble_mean = np.mean(tree_ensemble_scores)
print(f"  Mean: {tree_ensemble_mean:.4f} ± {np.std(tree_ensemble_scores):.4f}")

print("\n" + "="*60)
print("FINAL COMPARISON:")
print(f"  Random Forest alone:       {individual_scores['Random Forest']:.4f}")
print(f"  All 4 models ensemble:     {ensemble_mean:.4f}")
print(f"  Tree models only ensemble: {tree_ensemble_mean:.4f}")
print("="*60)


Testing ENSEMBLE (tree models only - no LogReg)...
  Fold 1: 0.5271
  Fold 2: 0.5222
  Fold 3: 0.5242
  Fold 4: 0.5271
  Fold 5: 0.5249
  Mean: 0.5251 ± 0.0018

FINAL COMPARISON:
  Random Forest alone:       0.5260
  All 4 models ensemble:     0.5245
  Tree models only ensemble: 0.5251


### Training only Random Forest on all data

In [11]:
# Train Random Forest on all data

print("Training final Random Forest on all data:")

final_rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    random_state=42
)

final_rf.fit(X_base[feat_cols], y)

# Predict on test
test_pred = final_rf.predict(X_test_base[feat_cols])

# Create submission
submission = pd.DataFrame({
    'ROW_ID': X_test.index,
    'prediction': test_pred
})

submission.to_csv('../data/randFor_sub.csv', index=False)

print("Random Forest submission created")
print(f"CV Score: {individual_scores['Random Forest']:.4f}")

Training final Random Forest on all data:
Random Forest submission created
CV Score: 0.5260
